# Development of Arctic boundary current model

See `Development/TestingNavierStokes??.ipynb` series for development of NS solver in rectangular domain.

0. `ArcticBoundaryCurrentModel0.ipynb`: Adapt wind forced dishpan steady NS solver to Arctic boundary current geometry.
1. `ArcticBoundaryCurrentModel1.ipynb`: More work on gmsh domain. 
2. `ArcticBoundaryCurrentModel2.ipynb`: More work on gmsh domain. Add cooling and freshening patches to cylinder.
3. `ArcticBoundaryCurrentModel3.ipynb`: More work on gmsh domain. Add warming/salinification patch. But it still errors when defining the 3D domain (it only works on the surface of the gmsh domain).
4.  `ArcticBoundaryCurrentModel4.ipynb`: More work on gmsh domain. Try to get cylindrical disk to work. This succeeds.
5.  `ArcticBoundaryCurrentModel5.ipynb`: More work on gmsh domain. Try to add cutoff cone beneath cylinder. This succeeds, except the gridap solution is zero in the cone.
6.  `ArcticBoundaryCurrentModel6.ipynb`: More work on gmsh domain. Try to join cylinder and cone beneath cylinder. See also `GridapGmsh_demo.ipynb`. This succeeds.
7.  `ArcticBoundaryCurrentModel7.ipynb`: More work on gmsh domain. Try to add freshening and cooling patches...this takes a long time to get the Gmsh .geo file working right. See Development/mwe_openCASCADE_{errors,works}.geo.
8.  `ArcticBoundaryCurrentModel8.ipynb`: More work on gmsh domain. Try to add warming/salinification patch. This succeeds.
9.  `ArcticBoundaryCurrentModel9.ipynb`: Experiment with simple forcing fields, parameters, and mesh options. ~34000 tetrahedra takes about 6.5 minutes to run for an O(1m) size domain and an Ekman depth of 0.2m and a uniform wind stress.
10.  `ArcticBoundaryCurrentModel10.ipynb`: Switch to azimuthal windstress with ~18k tetraheda (takes 2.5mins). Explore decreasing viscosity and boundary mesh only. The pressure field is very weak in 3D with large viscosity.
11.  `ArcticBoundaryCurrentModel11.ipynb`: Try very small viscosity and different domain aspect ratios, including thin disks and lots of meshes. For thin domains, large numbers of tetrahedra and/or small viscosity, the solution is vanishing flow. Why?  I'm thinking it may be due to wrong scaling, e.g., of the stabilization parameter $\alpha$.
12.  `ArcticBoundaryCurrentModel12.ipynb`: Switch to free-slip, impermeable velocity boundary conditions on the interior walls. The impermeability part is tricky because with an homogeneous Dirichlet bc, it's guaranteed. Now we want a non-zero tangential velocity at the wall. I.e., the bc on the normal flow at the wall is homogeneous Dirchlet, and the bc on the tangential flow component is homogeneous Neumann. Ask question [here](https://app.gitter.im/#/room/#Gridap-jl_community:gitter.im). Discover [`nuPGCM`](https://github.com/hgpeterson/nuPGCM) and investigate that as an option for the Arctic boundary current model. See the `nuPGCM_development` folder.


twnh July '25

## Problem statement

The ultimate goal is to solve a nonlinear multi-field PDE. Consider here the lid-stress-driven flow for the incompressible rotating Navier-Stokes equations. Formally, the PDE we want to solve is: find the velocity vector $u$ and the pressure anomaly $p$ such that

$$
\left\lbrace
\begin{aligned}
-\nu  \nabla^2 u +  (u \nabla) u + (1/\rho_0) \nabla p + f  \hat{\mathbf k} \times u = 0 &\text{ in }\Omega,\\
\nabla\cdot u = 0 &\text{ in } \Omega,\\
\boldsymbol{t} \cdot \boldsymbol{\sigma} \cdot \mathbf{n} = \tau_{\text{imposed}} &\text{ on } \Gamma_s, \\
\boldsymbol{t} \cdot \boldsymbol{\sigma} \cdot \mathbf{n} = \boldsymbol{0} &\text{ on } \Gamma_w, \\
u \cdot \boldsymbol{n} = 0 &\text{ on } \Gamma_w \cup \Gamma_s
\end{aligned}
\right.
$$

where the computational domain is the rectangle $\Omega \doteq (0,L_x) \times (-L_y/2,L_y/2) \times (-L_z,0)$, ${\mathbf n}$ is the unit outward normal, $\boldsymbol{t}$ is the tangential direction at the surface, $\Gamma_s$, and $\boldsymbol{\sigma} = \nabla u$ is the stress vector. The driving force is the tangential stress $\tau_{\text{imposed}}$ (units of $\text{m}^{2} \text{s}^{-2}$ ). The mean value of the pressure anomaly is constrained to equal zero,

$$
\int_\Omega p \ {\rm d}\Omega = 0 .
$$


In [9]:
using Gridap
using GridapSolvers
using GridapSolvers.LinearSolvers, GridapSolvers.MultilevelTools, GridapSolvers.NonlinearSolvers
using GridapSolvers.BlockSolvers: LinearSystemBlock, NonlinearSystemBlock, BiformBlock, BlockTriangularSolver
using Gridap.MultiField
using GridapGmsh
using LinearAlgebra

#### Define physical parameters

In [10]:
# Physical properties
# ν = 1e-6              # (m^2/s) kinematic viscosity
ν = 1.0e-2              # (m^2/s) kinematic viscosity
ρₒ = 1000.0             # (kg/m^3) reference density
rotation_period = 6.0   # (s) rotation rate
f = 2*2*π/rotation_period
Ekman_layer_depth = sqrt(2*ν / f) # (m), Ekman layer depth
println("Ekman layer depth: ", Ekman_layer_depth, " m")

Ekman layer depth: 0.097720502380584 m


#### Surface stress boundary condition

In [11]:
# u₁₀(y) = 1.0     # m s⁻¹, average wind velocity 10 meters above the ocean
# u₁₀(y) =  1.0 .* cos(π.*y./Ly)     # m s⁻¹, average wind velocity 10 meters above the ocean
function u₁₀(x_in)
    x,y = x_in[1], x_in[2]
    r = sqrt(x^2 + y^2)
    θ = atan(y, x)
    radial_wind = 1.0e-1 * r        # Set magnitude of wind velocity here
    u₁₀ = radial_wind * [ - sin(θ),  cos(θ), 0.0 ] # m s⁻¹
    return u₁₀
end

cᴰ = 2.5e-3 # dimensionless drag coefficient
ρₐ = 1.225  # kg m⁻³, average density of air at sea-level
Qᵘ(x) = VectorValue((ρₐ / ρₒ) * cᴰ * u₁₀(x) * norm(u₁₀(x))) # m² s⁻²

Qᵘ (generic function with 1 method)

#### Load model geometry.

In [12]:
model = GmshDiscreteModel("ArcticBasin12.msh")
boundaries = ["Slope", "Wall", "WarmingPatch", "Bottom", "FresheningPatch", "CoolingPatch"]
labels = get_face_labeling(model)
label_names = model.face_labeling.tag_to_name


Info    : Reading 'ArcticBasin12.msh'...
Info    : 32 entities
Info    : 3829 nodes
Info    : 19490 elements
Info    : Done reading 'ArcticBasin12.msh'


7-element Vector{String}:
 "Slope"
 "WarmingPatch"
 "Wall"
 "Bottom"
 "FresheningPatch"
 "CoolingPatch"
 "ArcticBasin"

## FE spaces

See: `Development_dont_delete/TestingNavierStokes??.ipynb` for more info on these spaces

In [13]:
order = 2
qdegree = 2*(order+1)
reffeᵤ = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
# V = TestFESpace(model,reffeᵤ,conformity=:H1,labels=labels,dirichlet_tags=["Wall","Bottom","Slope"])
V = TestFESpace(model,reffeᵤ,conformity=:H1,labels=labels)
reffeₚ = ReferenceFE(lagrangian,Float64,order-1;space=:P)
Q = TestFESpace(model,reffeₚ,conformity=:L2,constraint=:zeromean)
uD0 = VectorValue(0,0,0)            # Velocity vanishes on the boundaries except the top
# U = TrialFESpace(V,uD0)
U = TrialFESpace(V)
P = TrialFESpace(Q)
mfs = Gridap.MultiField.BlockMultiFieldStyle()
Y = MultiFieldFESpace([V, Q];style=mfs)
X = MultiFieldFESpace([U, P];style=mfs)

MultiFieldFESpace()

## Triangulation and integration quadrature

From the discrete model we can define the triangulation and integration measure

In [14]:
degree = order
Ω = Triangulation(model)
dΩ = Measure(Ω,qdegree)
Γs = BoundaryTriangulation(model,tags="CoolingPatch")
# Γ = BoundaryTriangulation(model,tags=boundaries)
dΓs = Measure(Γs,degree)

GenericMeasure()

## Steady nonlinear Navier-Stokes solver with Coriolis force.

Use a preconditioned FGMRES solver. See: https://gridap.github.io/GridapSolvers.jl/stable/Examples/NavierStokes/

In [15]:
khat = VectorValue(0,0,1)
Coriolis(u,v,dΩ) = ∫(f * cross(khat, u) ⋅ v)dΩ # Coriolis term
stress_bc((v,q),dΓ) =  ∫(- v ⋅ Qᵘ)dΓ    # Boundary condition for the velocity

α = 1.e2    # Stabilization parameter. What the units of α?  Dimensionless?  Should it be scaled with the density?  α appears in 2 places below...
Π_Qh = LocalProjectionMap(divergence,Q,qdegree)
graddiv(u,v,dΩ) = ∫(α*(∇⋅v)⋅Π_Qh(u))dΩ

conv(u,∇u) = (∇u')⋅u
dconv(du,∇du,u,∇u) = conv(u,∇du)+conv(du,∇u)
c(u,v,dΩ) = ∫(v⊙(conv∘(u,∇(u))))dΩ
dc(u,du,dv,dΩ) = ∫(dv⊙(dconv∘(du,∇(du),u,∇(u))))dΩ

lap(u,v,dΩ) = ∫(ν*∇(v)⊙∇(u))dΩ

jac_u(u,du,dv,dΩ) = lap(du,dv,dΩ) + dc(u,du,dv,dΩ) + graddiv(du,dv,dΩ)
jac_u(u,du,dv,dΩ) = lap(du,dv,dΩ) + graddiv(du,dv,dΩ) + Coriolis(du,dv,dΩ)
jac((u,p),(du,dp),(dv,dq),dΩ) = jac_u(u,du,dv,dΩ) - ∫(divergence(dv)*dp)dΩ - ∫(divergence(du)*dq)dΩ

res_u(u,v,dΩ) = lap(u,v,dΩ) + c(u,v,dΩ) + graddiv(u,v,dΩ)
res_u(u,v,dΩ) = lap(u,v,dΩ) + graddiv(u,v,dΩ) + Coriolis(u,v,dΩ) 
res((u,p),(v,q),dΩ) = res_u(u,v,dΩ) - (1/ρₒ)*∫(divergence(v)*p)dΩ - (1/ρₒ)*∫(divergence(u)*q)dΩ + stress_bc((v,q),dΓs)

jac_h(x,dx,dy) = jac(x,dx,dy,dΩ)
res_h(x,dy) = res(x,dy,dΩ)
op = FEOperator(res_h,jac_h,X,Y)

solver_u = LUSolver()
solver_p = CGSolver(JacobiLinearSolver();maxiter=20,atol=1e-14,rtol=1.e-6,verbose=true)
solver_p.log.depth = 4

bblocks  = [NonlinearSystemBlock() LinearSystemBlock();
            LinearSystemBlock()    BiformBlock((p,q) -> ∫(-(1/(ρₒ*ρₒ*α))*p*q)dΩ,Q,Q)]
            # LinearSystemBlock()    BiformBlock((p,q) -> ∫(-(1/α)*p*q)dΩ,Q,Q)]
coeffs = [1.0 1.0;
          0.0 1.0]  
P = BlockTriangularSolver(bblocks,[solver_u,solver_p],coeffs,:upper)
solver = FGMRESSolver(20,P;atol=1e-11,rtol=1.e-8,verbose=true)
solver.log.depth = 2

nlsolver = NewtonSolver(solver;maxiter=20,atol=1e-10,rtol=1.e-12,verbose=true)
uh,ph = solve(nlsolver,op)

--------------- Starting Newton-Raphson solver --------
  > Iteration   0 - Residuals: 1.72e-10,   1.00e+00 
    --------------- Starting FGMRES solver ----------------
      > Iteration   0 - Residuals: 1.72e-10,   1.00e+00 
        --------------- Starting CG solver --------------------
          > Iteration   0 - Residuals: 0.00e+00,   1.00e+00 
        Solver CG finished with reason SOLVER_CONVERGED_ATOL
        Iterations:   0 - Residuals: 0.00e+00,   NaN 
        --------------- Exiting CG solver ---------------------
      > Iteration   1 - Residuals: 4.02e-14,   2.33e-04 
    Solver FGMRES finished with reason SOLVER_CONVERGED_ATOL
    Iterations:   1 - Residuals: 4.02e-14,   2.33e-04 
    --------------- Exiting FGMRES solver -----------------
  > Iteration   1 - Residuals: 4.13e-17,   2.40e-07 
Solver Newton-Raphson finished with reason SOLVER_CONVERGED_ATOL
Iterations:   1 - Residuals: 4.13e-17,   2.40e-07 
--------------- Exiting Newton-Raphson solver ---------


MultiFieldFEFunction():
 num_fields: 2
 num_cells: 12728
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 10161879565937557555

Post-process and tidy up.

In [16]:
mag_integral = sqrt(sum( ∫( uh ⋅ uh )dΩ ))
domain_volume = sum(∫(1)dΩ)
avg_velocity_magnitude = mag_integral / domain_volume

println("Average |u| = ", avg_velocity_magnitude, " m/s. Domain volume = ", domain_volume, " m³.")

writevtk(Ω,"ArcticBoundaryCurrentModel12",cellfields=["uh"=>uh,"ph"=>ph])

Average |u| = 4.419849655396025e-8 m/s. Domain volume = 0.2679726769493389 m³.


(["ArcticBoundaryCurrentModel12.vtu"],)